# Tools 
## models can request tools to perform specific tasks such as fetching data from databases ,searching the web ,or running code . Tools are pairing of:

  ## 1. Tool name : A unique identifier for the tool.
  ## 2. Tool function : A callable function that implements the tool's functionality.

In [2]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-safeguard-20b
whisper-large-v3
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi


In [5]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("qwen/qwen3.8-27b", model_provider="groq", temperature=0, max_tokens=512)
response = model.invoke("write me a 10 paragraph on AI")
response 


AIMessage(content='Artificial Intelligence, or AI, has rapidly evolved from a theoretical concept in computer science to a transformative force that permeates nearly every aspect of modern life. At its core, AI refers to the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction. This technology is no longer confined to science fiction narratives; it is an integral part of the infrastructure that supports global communication, commerce, and healthcare. The journey of AI began in the mid-20th century with early experiments in logic and problem-solving, but it was the advent of big data and powerful computing hardware in the 21st century that truly accelerated its development.\n\nThe foundation of modern AI lies in machine learning, a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. Unlike traditional software, which follows a fixed set of rules, machine learning algorit

In [7]:
from langchain.tools import tool 

@tool 
def get_weather(location:str)-> str:
    """Get the weather of a location"""
    return f"The weather in {location} is sunny"

model_with_tools = model.bind_tools([get_weather])


In [8]:
response =model_with_tools.invoke("what is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool :{tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 't3gc5ytmy', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_time': 0.073545573, 'completion_tokens_details': None, 'prompt_time': 0.021031707, 'prompt_tokens_details': None, 'queue_time': 0.08248974, 'total_time': 0.09457728}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1d5c6-4a7f-74f7-ad32-c8454d1234fc-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 't3gc5ytmy', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 27, 'total_tokens': 305}
Tool :get_weather
Args: {'location': 'New York'}


#Tool Exceution Loops

In [13]:
#step 1: Model generate tool calls
messages=[{"role":"user","content":"what is the weather in New York?"}]
msg = model_with_tools.invoke(messages)
messages.append(msg)
#step2 : Excute tools and collect results
for tool_call in msg.tool_calls:
    #Excuete the tool with generated arguments 
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step3: Pass result back to model for final reespose 
final_response = model_with_tools.invoke(messages)
print(final_response)
#    

content='The weather in New York is currently sunny. ☀️' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 329, 'total_tokens': 342, 'completion_time': 0.02507882, 'completion_tokens_details': None, 'prompt_time': 0.024743477, 'prompt_tokens_details': None, 'queue_time': 0.054757842, 'total_time': 0.049822297}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1d5d7-0546-772c-a205-b08cbcd321ea-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 329, 'output_tokens': 13, 'total_tokens': 342}


In [14]:
messages

[{'role': 'user', 'content': 'what is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6et9xtwf4', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_time': 0.073478085, 'completion_tokens_details': None, 'prompt_time': 0.021075651, 'prompt_tokens_details': None, 'queue_time': 0.051208978, 'total_time': 0.094553736}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d5d7-0393-7147-a2c4-e586e8aae40f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '6et9xtwf4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 27, 'total_tokens': 305}),
 ToolMessage(content='The 